<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_02_target_definition/stage_02_01_target_investigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage 02 — CAP 1 - Target Architecture****


```
S02_C01_target_architecture.ipynb
```



## **Configuración del Entorno**


### **Acceso a Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


###**Importación de librerías**


In [ ]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

###**Definición de rutas**

In [ ]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [ ]:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/02_processed/mnq_intraday.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/state_02_target_investigation_summary.json"))

In [ ]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### **Función para ver información de dataset**


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")


### **Carga de dataset `intraday_mnq`**


In [ ]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [ ]:
mnq_intraday = load_mnq_parquet()

Archivo encontrado en disco. Cargando dataset local...


In [ ]:
info = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Dataset: mnq_intraday
Shape: (1024062, 8)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2026-04-17 16:00:00-04:00
First/Last day: 2020-01-02  ->  2026-04-17
Total days (trading): 1482
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2026-04-17 20:00:00+00:00


### **Validación temporal de dataset `mnq_intraday`**


Valida lo esencial para series temporales antes de construir targets: orden cronológico, duplicados, consistencia por día, monotonicidad de minute_of_day, gaps temporales y saltos sospechosos. Esto es importante porque en trading hay que validar cuidadosamente los timestamps para evitar look-ahead bias, y porque en series temporales el orden secuencial es parte central del problema.

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# Validación temporal de mnq_intraday
# Ejecutar inmediatamente después de:
# mnq_intraday = load_mnq_parquet()
# ============================================================

def validate_mnq_intraday(df: pd.DataFrame, verbose: bool = True) -> dict:
    """
    Valida consistencia temporal básica para un dataset intradía.

    Chequeos:
    1) Índice datetime válido
    2) Orden cronológico global
    3) Duplicados de timestamp
    4) Consistencia de columna `date`
    5) Monotonía de `minute_of_day` dentro de cada día
    6) Duplicados de `minute_of_day` dentro de cada día
    7) Saltos temporales negativos o nulos
    8) Gaps intradía distintos de 1 minuto
    """

    result = {
        "ok": True,
        "checks": {},
        "summary": {},
        "artifacts": {}
    }

    df = df.copy()

    # ------------------------------------------------------------
    # 0) Verificaciones básicas de estructura
    # ------------------------------------------------------------
    required_cols = ["date", "minute_of_day", "open", "high", "low", "close", "volume"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame debe ser un pd.DatetimeIndex")

    if df.empty:
        raise ValueError("El DataFrame está vacío")

    # ------------------------------------------------------------
    # 1) Orden global del índice
    # ------------------------------------------------------------
    is_monotonic = df.index.is_monotonic_increasing
    has_unique_index = df.index.is_unique
    duplicated_index = df.index[df.index.duplicated()].unique()

    result["checks"]["index_is_monotonic_increasing"] = bool(is_monotonic)
    result["checks"]["index_is_unique"] = bool(has_unique_index)
    result["summary"]["n_duplicated_timestamps"] = int(len(duplicated_index))
    result["artifacts"]["duplicated_timestamps"] = duplicated_index

    # Si no está ordenado, mostramos evidencia pero no reordenamos silenciosamente
    if not is_monotonic:
        diffs_ns = pd.Series(df.index.view("i8")).diff()
        bad_order_pos = np.where(diffs_ns <= 0)[0]
        result["artifacts"]["bad_global_order_positions"] = bad_order_pos[:20]
        result["ok"] = False

    if not has_unique_index:
        result["ok"] = False

    # ------------------------------------------------------------
    # 2) Consistencia entre index.date y columna `date`
    # ------------------------------------------------------------
    # Normalizamos ambos a fecha sin hora
    index_dates = pd.Index(df.index.tz_localize(None).date if df.index.tz is not None else df.index.date)
    col_dates = pd.to_datetime(df["date"]).dt.date

    date_match = (index_dates == col_dates).all()
    result["checks"]["date_column_matches_index_date"] = bool(date_match)

    if not date_match:
        mismatch_mask = index_dates != col_dates
        mismatches = df.loc[mismatch_mask, ["date", "minute_of_day", "close"]].head(20)
        result["artifacts"]["date_mismatches_head"] = mismatches
        result["summary"]["n_date_mismatches"] = int(mismatch_mask.sum())
        result["ok"] = False
    else:
        result["summary"]["n_date_mismatches"] = 0

    # ------------------------------------------------------------
    # 3) Diferencias temporales globales
    # ------------------------------------------------------------
    # Trabajamos en segundos
    diffs_sec = pd.Series(df.index).diff().dt.total_seconds()

    n_non_positive_diffs = int((diffs_sec.iloc[1:] <= 0).sum())
    result["checks"]["all_global_time_diffs_positive"] = (n_non_positive_diffs == 0)
    result["summary"]["n_non_positive_global_diffs"] = n_non_positive_diffs

    if n_non_positive_diffs > 0:
        bad_diff_rows = df.iloc[np.where((diffs_sec <= 0).fillna(False))[0][:20]]
        result["artifacts"]["non_positive_global_diffs_head"] = bad_diff_rows
        result["ok"] = False

    # ------------------------------------------------------------
    # 4) Validación por día
    # ------------------------------------------------------------
    daily_stats = []
    bad_minute_order_days = []
    duplicate_minute_days = []
    intraday_gap_rows = []

    grouped = df.groupby("date", sort=False)

    for day, g in grouped:
        g = g.copy()

        # 4.1 orden del índice dentro del día
        idx_mono = g.index.is_monotonic_increasing

        # 4.2 minute_of_day creciente dentro del día
        mod_diff = g["minute_of_day"].diff()
        minute_order_ok = bool((mod_diff.iloc[1:] > 0).all())

        # 4.3 duplicados de minute_of_day dentro del día
        dup_mod = g["minute_of_day"].duplicated().sum()
        has_dup_mod = dup_mod > 0

        # 4.4 gaps intradía del índice
        idx_diff_sec = pd.Series(g.index).diff().dt.total_seconds()
        gap_mask = (~idx_diff_sec.isna()) & (idx_diff_sec != 60)

        n_intraday_gaps = int(gap_mask.sum())

        if not minute_order_ok:
            bad_minute_order_days.append(day)

        if has_dup_mod:
            duplicate_minute_days.append(day)

        if n_intraday_gaps > 0:
            gap_info = g.loc[gap_mask, ["date", "minute_of_day", "open", "high", "low", "close", "volume"]].copy()
            gap_info["gap_seconds"] = idx_diff_sec[gap_mask].values
            intraday_gap_rows.append(gap_info)

        daily_stats.append({
            "date": day,
            "n_rows": len(g),
            "index_monotonic": bool(idx_mono),
            "minute_of_day_monotonic": minute_order_ok,
            "n_duplicate_minute_of_day": int(dup_mod),
            "n_intraday_gaps_not_60s": n_intraday_gaps,
            "minute_min": int(g["minute_of_day"].min()),
            "minute_max": int(g["minute_of_day"].max()),
        })

        if not idx_mono or not minute_order_ok or has_dup_mod:
            result["ok"] = False

    daily_stats_df = pd.DataFrame(daily_stats)

    result["artifacts"]["daily_stats"] = daily_stats_df
    result["summary"]["n_days"] = int(daily_stats_df.shape[0])
    result["summary"]["days_with_bad_minute_order"] = int(len(bad_minute_order_days))
    result["summary"]["days_with_duplicate_minute_of_day"] = int(len(duplicate_minute_days))
    result["summary"]["days_with_intraday_gaps_not_60s"] = int((daily_stats_df["n_intraday_gaps_not_60s"] > 0).sum())

    result["checks"]["all_days_have_monotonic_minute_of_day"] = (len(bad_minute_order_days) == 0)
    result["checks"]["no_duplicate_minute_of_day_within_day"] = (len(duplicate_minute_days) == 0)
    result["checks"]["all_intraday_steps_are_60s_within_day"] = bool(
        (daily_stats_df["n_intraday_gaps_not_60s"] == 0).all()
    )

    result["artifacts"]["bad_minute_order_days"] = bad_minute_order_days
    result["artifacts"]["duplicate_minute_days"] = duplicate_minute_days

    if intraday_gap_rows:
        result["artifacts"]["intraday_gaps_head"] = pd.concat(intraday_gap_rows, axis=0).head(50)
    else:
        result["artifacts"]["intraday_gaps_head"] = pd.DataFrame()

    # ------------------------------------------------------------
    # 5) Resumen global
    # ------------------------------------------------------------
    result["summary"]["n_rows"] = int(len(df))
    result["summary"]["start"] = df.index.min()
    result["summary"]["end"] = df.index.max()

    # ------------------------------------------------------------
    # 6) Reporte por pantalla
    # ------------------------------------------------------------
    if verbose:
        print("=" * 70)
        print("VALIDACIÓN TEMPORAL DE mnq_intraday")
        print("=" * 70)
        print(f"Rows                     : {result['summary']['n_rows']}")
        print(f"Days                     : {result['summary']['n_days']}")
        print(f"Start                    : {result['summary']['start']}")
        print(f"End                      : {result['summary']['end']}")
        print("-" * 70)
        print(f"Index monotonic          : {result['checks']['index_is_monotonic_increasing']}")
        print(f"Index unique             : {result['checks']['index_is_unique']}")
        print(f"Date == index.date       : {result['checks']['date_column_matches_index_date']}")
        print(f"Global diffs > 0         : {result['checks']['all_global_time_diffs_positive']}")
        print(f"minute_of_day monotonic  : {result['checks']['all_days_have_monotonic_minute_of_day']}")
        print(f"No dup minute_of_day     : {result['checks']['no_duplicate_minute_of_day_within_day']}")
        print(f"Intraday steps = 60s     : {result['checks']['all_intraday_steps_are_60s_within_day']}")
        print("-" * 70)
        print(f"Duplicated timestamps    : {result['summary']['n_duplicated_timestamps']}")
        print(f"Date mismatches          : {result['summary']['n_date_mismatches']}")
        print(f"Non-positive global diffs: {result['summary']['n_non_positive_global_diffs']}")
        print(f"Bad minute order days    : {result['summary']['days_with_bad_minute_order']}")
        print(f"Dup minute_of_day days   : {result['summary']['days_with_duplicate_minute_of_day']}")
        print(f"Days with !=60s gaps     : {result['summary']['days_with_intraday_gaps_not_60s']}")
        print("-" * 70)
        print(f"DATASET OK               : {result['ok']}")
        print("=" * 70)

        if result["summary"]["n_duplicated_timestamps"] > 0:
            print("\nDuplicated timestamps (head):")
            print(pd.Index(result["artifacts"]["duplicated_timestamps"][:10]))

        if result["summary"]["n_date_mismatches"] > 0:
            print("\nDate mismatches (head):")
            print(result["artifacts"]["date_mismatches_head"])

        if result["summary"]["days_with_bad_minute_order"] > 0:
            print("\nDays with bad minute_of_day order (head):")
            print(result["artifacts"]["bad_minute_order_days"][:10])

        if result["summary"]["days_with_duplicate_minute_of_day"] > 0:
            print("\nDays with duplicate minute_of_day (head):")
            print(result["artifacts"]["duplicate_minute_days"][:10])

        if not result["artifacts"]["intraday_gaps_head"].empty:
            print("\nIntraday gaps != 60 seconds (head):")
            print(result["artifacts"]["intraday_gaps_head"])

    return result


# ============================================================
# Ejecución inmediata
# ============================================================
validation = validate_mnq_intraday(mnq_intraday, verbose=True)

# Si quiere abortar automáticamente cuando haya problemas críticos:
critical_checks = [
    "index_is_monotonic_increasing",
    "index_is_unique",
    "date_column_matches_index_date",
    "all_global_time_diffs_positive",
    "all_days_have_monotonic_minute_of_day",
    "no_duplicate_minute_of_day_within_day",
]

failed_critical = [k for k in critical_checks if not validation["checks"].get(k, False)]

if failed_critical:
    raise ValueError(
        "Validación temporal fallida. Checks críticos con error: "
        + ", ".join(failed_critical)
    )

VALIDACIÓN TEMPORAL DE mnq_intraday
Rows                     : 1024062
Days                     : 1482
Start                    : 2020-01-02 04:30:00-05:00
End                      : 2026-04-17 16:00:00-04:00
----------------------------------------------------------------------
Index monotonic          : True
Index unique             : True
Date == index.date       : True
Global diffs > 0         : True
minute_of_day monotonic  : True
No dup minute_of_day     : True
Intraday steps = 60s     : True
----------------------------------------------------------------------
Duplicated timestamps    : 0
Date mismatches          : 0
Non-positive global diffs: 0
Bad minute order days    : 0
Dup minute_of_day days   : 0
Days with !=60s gaps     : 0
----------------------------------------------------------------------
DATASET OK               : True


El dataset `mnq_intraday` se encuentra correctamente estructurado para la construcción de targets temporales.

En particular, se verificó que:

- el índice `datetime` está ordenado cronológicamente en forma ascendente,
- no existen timestamps duplicados,
- la columna `date` coincide con la fecha derivada del índice,
- no hay saltos temporales negativos ni diferencias no positivas,
- dentro de cada jornada, `minute_of_day` es estrictamente creciente,
- no existen duplicados de `minute_of_day` dentro de un mismo día,
- y todos los pasos intradía son consistentes con una frecuencia de 1 minuto.

Por lo tanto, el dataset queda validado como temporalmente consistente y apto para construir targets forward del tipo `delta_h` y `ret_h`, siempre que dichos horizontes se calculen respetando los límites de cada jornada y evitando cruces entre días.

# **1. Introducción**


En el contexto de Machine Learning aplicado al trading algorítmico, la definición del target constituye una de las decisiones más importantes de todo el proceso de investigación. El target determina qué comportamiento del mercado intentará aprender el modelo y, en consecuencia, condiciona tanto la estructura de los datos como la interpretación y utilidad práctica de las predicciones obtenidas.

En problemas financieros, una aproximación natural consiste en intentar predecir retornos futuros de forma directa mediante modelos de regresión. Sin embargo, los retornos financieros presentan características que dificultan significativamente este enfoque, entre ellas:

* baja relación señal/ruido
* elevada volatilidad
* presencia de eventos extremos
* cambios de régimen de mercado
* baja estabilidad temporal de los patrones

Como resultado, pequeñas variaciones aleatorias pueden dominar el comportamiento de corto plazo del precio, dificultando el aprendizaje de relaciones robustas y generalizables.

Por este motivo, el enfoque moderno de Machine Learning for Trading tiende a reformular el problema hacia targets más estables y operativamente relevantes, priorizando problemas de clasificación y estructuras alineadas con decisiones reales de trading.  

Bajo esta perspectiva, el objetivo del modelo deja de ser únicamente anticipar movimientos exactos del mercado y pasa a centrarse en identificar escenarios con valor operativo, como movimientos direccionales relevantes, trades potencialmente exitosos o eventos asociados a reglas concretas de ejecución.

En este trabajo se propone una investigación jerárquica de distintos tipos de targets, comenzando desde formulaciones simples basadas en dirección del precio hasta estructuras más avanzadas orientadas a calidad de trade y eventos tipo barrera. Cada nuevo target busca corregir limitaciones del anterior, incrementando progresivamente el realismo operativo y la alineación con un sistema de trading real.

La investigación se estructura alrededor de cinco familias principales de targets:

* T1: dirección básica del movimiento
* T2: dirección con umbral de relevancia
* T3: outcome de trade
* T4: targets basados en eventos tipo barrera
* T5: formulaciones probabilísticas sobre los targets anteriores

El objetivo final no consiste únicamente en determinar cuál target presenta mejor capacidad predictiva, sino también evaluar cuál ofrece el mejor equilibrio entre robustez estadística, interpretabilidad y aplicabilidad operativa dentro de un sistema real de trading algorítmico.


# **2. Problema del retorno continuo**

Antes de definir targets de clasificación, resulta necesario analizar por qué no se utiliza directamente el retorno futuro continuo como variable objetivo del modelo. En términos generales, este enfoque consistiría en intentar predecir:

$$
r_{t,H} = \frac{P_{t+H} - P_t}{P_t}
$$

donde $P_t$ representa el precio en el instante actual y $P_{t+H}$ el precio luego de un horizonte futuro $H$.

A primera vista, esta formulación parece natural, ya que el retorno constituye la magnitud financiera de interés en cualquier estrategia de trading. Sin embargo, en la práctica, modelar retornos continuos presenta importantes dificultades estadísticas y operativas.

Uno de los principales problemas es la elevada relación señal/ruido presente en los mercados financieros. Gran parte de las variaciones de corto plazo del precio responden a fluctuaciones aleatorias, microestructura de mercado, noticias inesperadas o movimientos transitorios difíciles de modelar de manera consistente. Como consecuencia, el componente verdaderamente predecible suele ser pequeño en comparación con la variabilidad total observada.

Además, los retornos financieros rara vez siguen distribuciones normales estables. Es frecuente observar:

* colas pesadas
* eventos extremos
* volatilidad cambiante
* heterocedasticidad
* cambios de régimen de mercado

Estas características dificultan el aprendizaje de relaciones robustas y generan targets altamente inestables.

Otro problema importante es la baja persistencia temporal de muchas relaciones predictivas. Señales que parecen funcionar en determinados períodos pueden deteriorarse rápidamente fuera de muestra debido a cambios estructurales del mercado, comportamiento de participantes o condiciones de volatilidad. Esto provoca modelos con bajo poder de generalización y elevado riesgo de overfitting.

Desde el punto de vista operativo, predecir un retorno exacto tampoco necesariamente coincide con la lógica real de toma de decisiones en trading. En muchos casos, no resulta relevante estimar si el retorno futuro será exactamente (0.42%) o (0.57%), sino determinar si existe una oportunidad suficientemente favorable para justificar una operación.

Por este motivo, gran parte de los enfoques modernos de Machine Learning aplicado al trading reformulan el problema hacia estructuras de clasificación o eventos discretos, buscando targets más robustos, interpretables y alineados con decisiones reales de ejecución.

Bajo esta perspectiva, la evolución desde T1 hasta T4 puede interpretarse como una transición progresiva desde una representación simple del movimiento futuro del precio hacia formulaciones cada vez más cercanas a la lógica operativa de un sistema real de trading.


# **3. Construcción jerárquica de targets**

La construcción de targets en Machine Learning aplicado al trading puede entenderse como una evolución progresiva desde formulaciones simples y abstractas hacia representaciones más cercanas a la lógica operativa real de un sistema de trading.

Cada nuevo target surge como una respuesta a limitaciones detectadas en el anterior, buscando mejorar aspectos como:

- robustez estadística
- relación señal/ruido
- interpretabilidad
- alineación con decisiones reales de trading
- representación del riesgo y la ejecución

Bajo esta perspectiva, los targets utilizados en esta investigación no son independientes entre sí, sino que forman una estructura jerárquica donde cada nivel incorpora información adicional y una lógica operativa más compleja.

## **3.1. T1 — Dirección del movimiento**


**Definición conceptual**

El target T1 representa la formulación más simple del problema predictivo. El objetivo consiste únicamente en determinar si el precio futuro será mayor o no respecto al precio actual dentro de un horizonte temporal definido.

En este enfoque, el problema deja de formularse como una regresión sobre retornos continuos y pasa a convertirse en un problema de clasificación binaria.

---

**Fórmula**

El retorno *forward* para un horizonte \(H\) se define como:

$$
r_{t,H} = \frac{P_{t+H} - P_t}{P_t}
$$

donde:

- $P_t$: precio en el instante $t$
- $P_{t+H}$: precio futuro luego de un horizonte $H$

A partir de este retorno, el target binario $T1$ se define como:

$$
T1 =
\begin{cases}
1, & \text{si } r_{t,H} > 0 \\
0, & \text{si } r_{t,H} \leq 0
\end{cases}
$$

De esta forma:

- $T1 = 1$ indica un movimiento alcista.
- $T1 = 0$ indica un movimiento bajista o no alcista.
---

**Interpretación**

El modelo intenta responder únicamente:

> “¿El precio subirá dentro del horizonte definido?”

No importa la magnitud exacta del movimiento, sino únicamente su dirección.

---

**Ventajas**

* formulación simple
* fácil interpretación
* reducción parcial del ruido presente en retornos continuos
* baseline natural para modelos de clasificación
* entrenamiento relativamente estable

---

**Limitaciones**

* considera movimientos pequeños e irrelevantes como señales válidas
* mantiene elevada sensibilidad al ruido de mercado
* no incorpora criterios operativos
* ignora costos, volatilidad y calidad del movimiento
* puede generar señales con escaso valor económico

Estas limitaciones motivan la evolución hacia targets más selectivos.



## **3.2. T2 — Dirección con umbral**


**Definición conceptual**

El target T2 surge como una extensión natural de T1. Su objetivo consiste en eliminar movimientos pequeños o ambiguos que probablemente no posean valor operativo real.

En lugar de clasificar cualquier variación del precio como señal direccional, T2 incorpora un umbral mínimo de relevancia.

De esta manera, el modelo comienza a diferenciar entre:

* movimientos significativos
* movimientos insignificantes
* ausencia de señal clara

---

**Fórmula**

Sea $\theta$ un umbral mínimo de movimiento:

$$
T2 =
\begin{cases}
1, & \text{si } r_{t,H} > \theta \\
-1, & \text{si } r_{t,H} < -\theta \\
0, & \text{si } |r_{t,H}| \leq \theta
\end{cases}
$$

donde:

- $1$: movimiento alcista significativo
- $-1$: movimiento bajista significativo
- $0$: movimiento no significativo
---

**Relación con T1**

T2 reutiliza exactamente el mismo retorno futuro utilizado por T1, pero introduce una discretización más estricta mediante un filtro de relevancia.

Conceptualmente:

* T1 pregunta si el precio sube o baja
* T2 pregunta si el movimiento es suficientemente relevante para ser considerado una señal

---

**Ventajas**

* mejora la relación señal/ruido
* elimina movimientos marginales
* reduce ambigüedad del target
* genera señales potencialmente más operables
* introduce una noción inicial de valor económico

---

**Problemas que corrige**

Respecto a T1, T2 corrige principalmente:

* sobre-sensibilidad a fluctuaciones pequeñas
* exceso de señales débiles
* clasificación de ruido como información útil
* baja relevancia económica de ciertos movimientos

Sin embargo, todavía continúa modelando únicamente el resultado final del precio y no la calidad operativa de una operación real.


## **3.3. T3 — Outcome de trade**


**Definición conceptual**

Con T3 ocurre un cambio conceptual importante dentro de la investigación.

El objetivo deja de ser predecir el comportamiento del mercado de manera abstracta y pasa a centrarse directamente en evaluar la calidad de una posible operación.

En este enfoque, el modelo ya no intenta responder:

> “¿Subirá el precio?”

sino:

> “¿Este trade habría sido exitoso?”

El target comienza entonces a incorporar elementos propios de ejecución y gestión operativa.

---

**Componentes principales**

La construcción de T3 depende de:

* precio de entrada
* dirección del trade
* take profit (TP)
* stop loss (SL)
* horizonte temporal máximo

---

**Formulación conceptual**

El target $T3$ se define en función del resultado operativo de un trade:



$$
T3 =
\begin{cases}
1, & \text{trade exitoso} \\
0, & \text{trade no exitoso}
\end{cases}
$$

La definición exacta de éxito dependerá de las reglas operativas adoptadas para la estrategia.

---

**Interpretación**

T3 ya no modela únicamente movimientos del precio, sino escenarios operativos concretos.

Como consecuencia:

* el target se acerca significativamente a un sistema real de trading
* las predicciones se vuelven más interpretables desde el punto de vista operativo
* la evaluación del modelo comienza a alinearse con métricas reales de performance

## **3.4. T4 — Event-based target**


**Definición conceptual**

El target T4 representa una evolución adicional hacia una representación explícita de eventos de mercado y lógica de ejecución.

A diferencia de T1, T2 y parte de T3, T4 ya no depende únicamente del retorno final observado al final del horizonte.

Ahora importa el recorrido completo del precio dentro del horizonte futuro.

Este enfoque introduce el concepto de:

* triple barrier
* path dependency
* eventos de ejecución

---

**Formulación conceptual**

El target $T4$ se define en función del primer evento alcanzado por el precio:

$$
T4 =
\begin{cases}
1, & \text{TP alcanzado primero} \\
-1, & \text{SL alcanzado primero} \\
0, & \text{ninguna barrera alcanzada}
\end{cases}
$$

donde:

- $TP$: *Take Profit*
- $SL$: *Stop Loss*

---

**Interpretación**

El target ya no depende solamente de dónde termina el precio, sino de:

* qué evento ocurre primero
* cómo evoluciona el precio en el tiempo
* qué barrera se alcanza antes

Esto aproxima mucho más el problema a la lógica real de ejecución utilizada en trading algorítmico.

---

**Ventajas**

* modela explícitamente riesgo y recompensa
* incorpora dinámica temporal del precio
* reduce inconsistencias de targets basados solo en retorno final
* mejora alineación con backtesting y ejecución real

T4 constituye una de las representaciones más operativamente realistas dentro de targets supervisados para trading.

## **3.5. T5 — Probabilidades**


**Definición conceptual**

El target T5 no representa un target independiente como los anteriores.

En realidad, corresponde a una capa probabilística aplicada sobre cualquiera de los targets previos.

En lugar de producir únicamente una clase discreta, el modelo estima probabilidades condicionales:

$$
P(y=k|x)
$$

---

**Interpretación**

El objetivo pasa de:

> “predecir una clase”

a:

> “estimar el grado de confianza asociado a cada clase”

Esto permite:

* filtrar señales débiles
* controlar agresividad operativa
* operar únicamente escenarios de alta convicción
* optimizar precision vs trade frequency

---

**Relación con targets anteriores**

T5 no reemplaza:

* T1
* T2
* T3
* T4

sino que agrega una capa adicional de decisión sobre ellos.

Por ejemplo:

* T1 puede convertirse en clasificación probabilística binaria
* T2 en clasificación probabilística multiclase
* T4 en probabilidad de TP vs SL

---

**Importancia operativa**

Las probabilidades constituyen uno de los elementos más importantes dentro de sistemas reales de trading basados en ML, ya que permiten transformar señales predictivas en reglas de decisión adaptativas y controladas por nivel de confianza.


# **4. Relación jerárquica entre targets**

## **4.1. Evolución conceptual de los targets**


Los targets definidos en esta investigación no representan formulaciones aisladas, sino una evolución progresiva hacia representaciones cada vez más cercanas a la lógica operativa real del trading algorítmico.

Cada nuevo target surge como una extensión del anterior, incorporando nuevos elementos destinados a corregir limitaciones previamente identificadas. En consecuencia, la secuencia T1–T5 puede interpretarse como una construcción jerárquica donde aumenta progresivamente:

* la complejidad del problema
* el realismo operativo
* la incorporación explícita del riesgo
* la alineación con decisiones reales de ejecución

Bajo esta perspectiva, la investigación no busca únicamente comparar targets independientes, sino analizar cómo distintas formulaciones del problema modifican la capacidad predictiva y operativa de los modelos de Machine Learning.


## **4.2. Tabla jerárquica de targets**

| Target | Construido a partir de     | Objetivo principal                  |
| ------ | -------------------------- | ----------------------------------- |
| T1     | retorno futuro             | dirección del movimiento            |
| T2     | T1 + umbral                | filtrar ruido y movimientos débiles |
| T3     | T2 + lógica operativa      | evaluar calidad del trade           |
| T4     | T3 + barreras/eventos      | modelar ejecución real              |
| T5     | probabilidades sobre T1–T4 | estimar confianza de la señal       |


## **4.3. Interpretación jerárquica**


La transición desde T1 hacia T4 representa un desplazamiento progresivo desde targets puramente estadísticos hacia formulaciones explícitamente operativas.

En T1 y T2, el modelo todavía intenta anticipar movimientos futuros del mercado utilizando únicamente información derivada del retorno forward. En cambio, T3 y T4 incorporan directamente elementos asociados a la lógica de ejecución, gestión del riesgo y evaluación de operaciones concretas.

Finalmente, T5 introduce una capa probabilística transversal que permite transformar las predicciones discretas en estimaciones de confianza, facilitando la construcción de reglas de decisión adaptativas y sistemas de filtrado de señales.

Esta estructura jerárquica servirá como marco conceptual para toda la investigación posterior, organizando tanto el análisis estadístico de los targets como la evaluación predictiva y operativa de los distintos modelos de Machine Learning.

# **5. Comparación conceptual entre targets**

La evolución jerárquica presentada anteriormente también implica diferencias importantes entre los distintos targets en términos de complejidad, ruido estadístico y alineación operativa.

Mientras los primeros targets priorizan simplicidad y facilidad de modelado, los targets más avanzados incorporan progresivamente elementos asociados a ejecución, riesgo y comportamiento dinámico del mercado. Como consecuencia, aumenta tanto el realismo operativo como la complejidad del problema predictivo.

La siguiente tabla resume conceptualmente estas diferencias:

| Target | Complejidad | Nivel de ruido          | Realismo operativo |
| ------ | ----------- | ----------------------- | ------------------ |
| T1     | baja        | alto                    | bajo               |
| T2     | baja-media  | medio                   | medio              |
| T3     | media       | menor                   | alto               |
| T4     | alta        | menor                   | muy alto           |
| T5     | transversal | depende del target base | muy alto           |

Desde esta perspectiva, no existe un target universalmente superior. Cada formulación presenta ventajas y limitaciones dependiendo del objetivo del sistema de trading, el horizonte temporal, la frecuencia operativa y la capacidad predictiva del modelo utilizado.

Los targets más simples suelen facilitar el entrenamiento y la estabilidad estadística, pero pueden resultar insuficientes desde el punto de vista operativo. Por el contrario, los targets más avanzados ofrecen una representación más realista del proceso de trading, aunque introducen una mayor complejidad estructural y mayores exigencias de modelado.

En consecuencia, una parte central de esta investigación consistirá en analizar el equilibrio entre:

* predictibilidad
* robustez estadística
* interpretabilidad
* aplicabilidad operativa

para determinar qué formulaciones de target resultan más adecuadas dentro de un sistema real de Machine Learning aplicado al trading algorítmico.


# **6. Conclusión del capítulo**


La construcción de targets constituye uno de los elementos centrales dentro del proceso de diseño de modelos de Machine Learning aplicados al trading algorítmico. A lo largo de este capítulo se presentó una estructura jerárquica de targets que evoluciona progresivamente desde formulaciones simples basadas únicamente en dirección del precio hacia representaciones cada vez más cercanas a la lógica operativa real del trading.

En este contexto, cada nuevo target surge como una extensión del anterior, incorporando mecanismos destinados a reducir limitaciones asociadas al ruido financiero, la baja relevancia económica de ciertos movimientos y la desconexión entre predicción estadística y ejecución operativa.

La secuencia T1–T5 permite observar esta transición de manera gradual:

* T1 introduce una formulación direccional básica
* T2 incorpora filtros de relevancia
* T3 modela directamente la calidad de una operación
* T4 incorpora eventos y path dependency
* T5 agrega una capa probabilística orientada a la confianza de las señales

Sin embargo, esta evolución también implica un incremento progresivo en la complejidad del problema y en las exigencias del modelado predictivo. Como consecuencia, no existe un target universalmente superior, sino distintos compromisos entre simplicidad, robustez estadística y realismo operativo.

En consecuencia, la investigación posterior buscará determinar qué formulaciones ofrecen el mejor equilibrio entre:

* capacidad predictiva
* robustez estadística
* interpretabilidad
* aplicabilidad operativa

dentro de un sistema real de Machine Learning orientado al trading algorítmico.

A partir de este marco conceptual, el siguiente capítulo abordará la construcción práctica y el análisis estadístico de los distintos targets definidos en esta investigación.
